# Mean-reversion within an uptrend (Z-score) on a halal universe

A more statistically-framed cousin of the RSI pullback notebook: instead of an oscillator, this
measures exactly how many standard deviations a stock's price has strayed from its own short-term
average, and buys the extreme dips — but only inside a confirmed long-term uptrend. Tickers come
from `equities-list.csv` (rows with `Asset Type == Stock`; the ETFs in that file are excluded).
The CSV labels these as halal *candidates* only — independently verify current Shariah compliance
before investing.

**Strategy specification**

$$ Z_t = \frac{P_t - \text{MA}_t}{\sigma_t} $$

where MA and &sigma; are both the 20-day rolling mean and standard deviation of price (the same
window Bollinger Bands use).

- **Entry** — buy when *both* hold on the same day: price is above its 200-day SMA (long-term
  uptrend intact), *and* $Z_t < -2$ (price has fallen more than two standard deviations below its
  own 20-day mean — an unusually large short-term decline). Unlike the RSI pullback notebook,
  there's no separate confirmation step: this buys directly into the extreme, on the theory that a
  2&sigma; dip inside an uptrend is itself the signal, not a warning to wait for a reversal to
  start.
- **Exit** — sell once $Z_t > 0$. Since $Z_t > 0 \iff P_t > \text{MA}_t$, this is *exactly* the same
  condition as "price returns to its 20-day SMA" — the two exit rules the spec offers aren't
  alternatives, they're the same rule written two ways.
- **Position management** — long only, one position per name at a time, up to 15 concurrent
  positions, each sized as an equal share of current portfolio value at entry. When more than 15
  names qualify on the same day, the most extreme (most negative Z) get the open slots first.
- **No protective stop** — deliberately, per spec: this strategy's only exit is mean reversion.
  See the caveats section for what that implies about tail risk on any single trade.

The spec calls out liquid mega-caps — NVDA, AMD, AVGO, MU, TSM, GOOGL, AMZN, META, MSFT — as
particularly interesting for this method, since the whole idea is to exploit a temporary
statistical deviation rather than to predict anything about long-term fundamentals, and those
are about as liquid and heavily-arbitraged as this universe gets. The backtest below still runs
across the full halal candidate universe (for consistency with the other notebooks), with a
dedicated section pulling out just those nine names' trades for a closer look.

Like the breakout and RSI pullback notebooks, this trades on a signal evaluated every hourly bar rather than on a monthly
calendar, so it runs the same kind of bar-by-bar event-driven simulation, reusing the shared
`data/hourly/{TICKER}.csv` cache. Every look-back keeps its trading-day length, converted to bars
at 7 per session.


## Before running

1. Install dependencies: `pip install ib-insync pandas numpy matplotlib pandas-datareader`
2. Run IBKR Trader Workstation or IB Gateway, enable API connections, and use a market-data-enabled
   account.
3. Set the host, port, and client ID below. Typical paper ports are 7497 for TWS and 4002 for
   Gateway; live ports are commonly 7496 and 4001.
4. Run the acquisition cell once. Each ticker's hourly bars are cached to
   `data/hourly/{TICKER}.csv`, and a summary of what downloaded is written to
   `data/hourly_download_log.csv`. All seven strategy notebooks share this cache. Re-running skips
   tickers that are already cached and logged `ok`, so it is safe to re-run if it is interrupted
   partway through.
5. IBKR only serves a limited span per hourly request, so each ticker is fetched backward in
   1-year chunks (about 18 requests per ticker back to 2009, roughly 2,400 for the full
   universe). With the default sleeps between requests, expect the first full download to take
   on the order of two hours; every notebook after that reads straight from the cache.


In [ ]:
from pathlib import Path
import csv
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from ib_insync import IB, Stock, util
except ImportError as exc:
    raise ImportError('Install ib-insync first: pip install ib-insync') from exc

EQUITIES_LIST_PATH = Path('equities-list.csv')
DATA_DIR = Path('data') / 'hourly'
LOG_PATH = Path('data') / 'hourly_download_log.csv'

with open(EQUITIES_LIST_PATH, newline='') as f:
    equities_rows = list(csv.DictReader(f))
UNIVERSE = [row['Ticker'] for row in equities_rows if row['Asset Type'] == 'Stock']
print(f'{len(UNIVERSE)} candidate halal stocks in the universe')

EXCHANGE = 'SMART'
CURRENCY = 'USD'
START_DATE = pd.Timestamp('2009-01-01', tz='UTC')
END_DATE = pd.Timestamp.now(tz='UTC')
BAR_SIZE = '1 hour'
DURATION = '1 Y'  # hourly history is fetched in 1-year chunks, walking backward from today
WHAT_TO_SHOW = 'TRADES'
USE_RTH = True

IB_HOST = '172.27.224.1'  # '127.0.0.1'
IB_PORT = 7497
IB_CLIENT_ID = 37
PACING_SLEEP_SECONDS = 3.0   # between tickers
CHUNK_SLEEP_SECONDS = 1.5    # between 1-year chunk requests
# Hourly bar geometry: US regular hours = 09:30 half-hour bar + six 1-hour bars = 7 bars/session.
BARS_PER_SESSION = 7
TRADING_DAYS_PER_YEAR = 252
BARS_PER_YEAR = BARS_PER_SESSION * TRADING_DAYS_PER_YEAR   # annualization factor for hourly returns
# Look-backs are written in trading days (each strategy's real-time horizon) and converted to bars
# with this factor. Set it to 1 to read every "day" window as a raw bar count instead (e.g. a
# 200-bar SMA), which turns the strategy into a much faster, shorter-horizon variant.
WINDOW_BARS_PER_DAY = BARS_PER_SESSION

# Strategy parameters
SMA_TREND_DAYS = 200      # long-term uptrend filter, in trading days
Z_WINDOW_DAYS = 20        # window for both the mean and the std in the Z-score, in trading days
SMA_TREND_BARS = SMA_TREND_DAYS * WINDOW_BARS_PER_DAY
Z_WINDOW_BARS = Z_WINDOW_DAYS * WINDOW_BARS_PER_DAY
Z_ENTRY_THRESHOLD = -2.0  # buy when Z drops below this
MAX_POSITIONS = 15

FOCUS_TICKERS = ['NVDA', 'AMD', 'AVGO', 'MU', 'TSM', 'GOOGL', 'AMZN', 'META', 'MSFT']

INVESTMENT_2026_DOLLARS = 150.0
TARGET_CPI = None  # leave None to use the latest available CPI reading from FRED

## Download or load the hourly bars

Hourly bars (unlike daily bars) can't be requested many years back in a single IBKR call, so each
ticker is walked backward in 1-year chunks from today until `START_DATE` or until IBKR has no
earlier data (a recent IPO simply stops early — check `first_date` in the download log). The
function still checks the on-disk cache first and only opens an IBKR connection if at least one
ticker's cache is missing. Bars are regular-trading-hours only: a 09:30 half-hour bar plus six
1-hour bars, i.e. 7 bars per full session.


In [ ]:
def normalize_bars(bars):
    frame = util.df(bars)
    if frame is None or frame.empty:
        return pd.DataFrame()
    frame['date'] = pd.to_datetime(frame['date'], utc=True)
    frame = frame.set_index('date').sort_index()
    keep = ['open', 'high', 'low', 'close', 'volume', 'average', 'barCount']
    return frame[[column for column in keep if column in frame.columns]]

def load_cached_bars(ticker):
    frame = pd.read_csv(DATA_DIR / f'{ticker}.csv', parse_dates=['date'], index_col='date')
    frame.index = pd.DatetimeIndex(frame.index).tz_convert('UTC')
    return frame.sort_index()

def fetch_ticker_hourly(ib, contract):
    """Walk backward in 1-year chunks until START_DATE or until IBKR has no earlier bars."""
    chunks, cursor, empty_in_a_row = [], END_DATE, 0
    while cursor > START_DATE:
        bars = ib.reqHistoricalData(
            contract, endDateTime=cursor.strftime('%Y%m%d %H:%M:%S UTC'), durationStr=DURATION,
            barSizeSetting=BAR_SIZE, whatToShow=WHAT_TO_SHOW,
            useRTH=USE_RTH, formatDate=2, keepUpToDate=False,
        )
        chunk = normalize_bars(bars)
        if chunk.empty:
            empty_in_a_row += 1
            if chunks or empty_in_a_row >= 2:  # ran out of history, or nothing recent at all
                break
            cursor -= pd.Timedelta(days=365)
        else:
            chunks.append(chunk)
            empty_in_a_row = 0
            cursor = chunk.index.min() - pd.Timedelta(hours=1)
        time.sleep(CHUNK_SLEEP_SECONDS)
    if not chunks:
        return pd.DataFrame()
    result = pd.concat(chunks).sort_index()
    result = result.loc[~result.index.duplicated(keep='last')]
    return result.loc[result.index >= START_DATE]

def download_universe():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    util.patchAsyncio()
    ib = IB()
    ib.connect(IB_HOST, IB_PORT, clientId=IB_CLIENT_ID, readonly=True, timeout=20)

    log_rows = list(csv.DictReader(open(LOG_PATH))) if LOG_PATH.exists() else []
    done = {row['ticker'] for row in log_rows if row['status'] == 'ok'}

    for i, ticker in enumerate(UNIVERSE, 1):
        cache_path = DATA_DIR / f'{ticker}.csv'
        if ticker in done and cache_path.exists():
            continue
        status, n_bars, first_date, last_date, error = 'ok', 0, '', '', ''
        try:
            contract = Stock(ticker, EXCHANGE, CURRENCY)
            qualified = ib.qualifyContracts(contract)
            if not qualified or contract.conId == 0:
                raise RuntimeError('contract could not be qualified')
            frame = fetch_ticker_hourly(ib, contract)
            if frame.empty:
                status = 'no_data'
            else:
                frame.to_csv(cache_path, index_label='date')
                n_bars = len(frame)
                first_date, last_date = str(frame.index.min().date()), str(frame.index.max().date())
        except Exception as exc:
            status, error = 'error', str(exc)
        print(f'[{i}/{len(UNIVERSE)}] {ticker}: {status} n_bars={n_bars} {first_date}..{last_date} {error}')
        log_rows = [r for r in log_rows if r['ticker'] != ticker]
        log_rows.append({'ticker': ticker, 'status': status, 'n_bars': n_bars,
                          'first_date': first_date, 'last_date': last_date, 'error': error})
        with open(LOG_PATH, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['ticker', 'status', 'n_bars', 'first_date', 'last_date', 'error'])
            writer.writeheader()
            writer.writerows(log_rows)
        time.sleep(PACING_SLEEP_SECONDS)

    ib.disconnect()

missing = [t for t in UNIVERSE if not (DATA_DIR / f'{t}.csv').exists()]
if missing:
    print(f'{len(missing)} tickers not yet cached, downloading from IBKR...')
    download_universe()
else:
    print('All tickers already cached.')


## Build the close panel and the Z-score

The 200-day trend filter (1,400 bars), and the Z-score itself — a 20-day (140-bar) rolling mean and
standard deviation of hourly closes, computed vectorized across the whole universe at once. The
windows are the same number of trading days as the daily version, expressed in bars.


In [ ]:
hourly_closes = {}
for ticker in UNIVERSE:
    cache_path = DATA_DIR / f'{ticker}.csv'
    if not cache_path.exists():
        continue
    bars = load_cached_bars(ticker)
    if bars.empty:
        continue
    hourly_closes[ticker] = bars['close']

close_panel = pd.DataFrame(hourly_closes).sort_index()
close_panel = close_panel.loc[close_panel.index >= START_DATE]
# Fill isolated missing bars (at most one session) so bar-count rolling windows stay contiguous.
close_panel = close_panel.ffill(limit=BARS_PER_SESSION)
print(f'{close_panel.shape[1]} tickers, {close_panel.shape[0]:,} hourly bars '
      f'from {close_panel.index.min().date()} to {close_panel.index.max().date()}')

sma200 = close_panel.rolling(SMA_TREND_BARS, min_periods=SMA_TREND_BARS).mean()
sma20 = close_panel.rolling(Z_WINDOW_BARS, min_periods=Z_WINDOW_BARS).mean()
std20 = close_panel.rolling(Z_WINDOW_BARS, min_periods=Z_WINDOW_BARS).std()
zscore = (close_panel - sma20) / std20


## Hourly event-driven backtest

Simpler than the RSI pullback notebook's state machine: entry is a direct same-bar condition
(price above its 200-day SMA and Z below -2), so there's no armed/confirmation step to track —
just exits first, then fill open slots with this bar's most extreme qualifying dips.


In [ ]:
dates = close_panel.index
tickers = close_panel.columns.tolist()
ticker_idx = {t: j for j, t in enumerate(tickers)}
n_dates = len(dates)

close_v = close_panel.values
sma200_v = sma200.values
z_v = zscore.values

positions = {}
cash = 1.0
nav_history = np.empty(n_dates)
trade_log = []

for i in range(n_dates):
    # 1. Exits: Z-score back above 0, i.e. price back above its own 20-day mean.
    to_remove = []
    for tkr, pos in positions.items():
        j = ticker_idx[tkr]
        c, z = close_v[i, j], z_v[i, j]
        if np.isnan(c) or np.isnan(z):
            continue
        if z > 0:
            cash += pos['shares'] * c
            trade_log.append({
                'ticker': tkr, 'entry_date': pos['entry_date'], 'entry_price': pos['entry_price'],
                'exit_date': dates[i], 'exit_price': c,
                'return_pct': (c / pos['entry_price'] - 1.0) * 100,
                'holding_days': (dates[i] - pos['entry_date']).total_seconds() / 86400,
            })
            to_remove.append(tkr)
    for tkr in to_remove:
        del positions[tkr]

    open_value = sum(positions[t]['shares'] * close_v[i, ticker_idx[t]] for t in positions
                      if not np.isnan(close_v[i, ticker_idx[t]]))
    nav_at_entry_phase = cash + open_value

    # 2. Entries: uptrend + Z < -2, most extreme dips first when slots are scarce.
    free_slots = MAX_POSITIONS - len(positions)
    if free_slots > 0:
        candidates = []
        for j, tkr in enumerate(tickers):
            if tkr in positions:
                continue
            c, s200, z = close_v[i, j], sma200_v[i, j], z_v[i, j]
            if np.isnan(c) or np.isnan(s200) or np.isnan(z):
                continue
            if c > s200 and z < Z_ENTRY_THRESHOLD:
                candidates.append((z, tkr, j))
        candidates.sort(key=lambda x: x[0])
        for z, tkr, j in candidates[:free_slots]:
            if cash <= 0:
                break  # no cash left to deploy this day, regardless of open slots
            alloc = min(nav_at_entry_phase / MAX_POSITIONS, cash)
            entry_price = close_v[i, j]
            shares = alloc / entry_price
            positions[tkr] = {'shares': shares, 'entry_price': entry_price, 'entry_date': dates[i]}
            cash -= alloc

    open_value = sum(positions[t]['shares'] * close_v[i, ticker_idx[t]] for t in positions
                      if not np.isnan(close_v[i, ticker_idx[t]]))
    nav_history[i] = cash + open_value

nav = pd.Series(nav_history, index=dates)
trades = pd.DataFrame(trade_log)
print(f'{len(trades):,} completed trades, {len(positions)} open positions as of {dates[-1].date()}')

### Current open positions

In [ ]:
last_i = n_dates - 1
open_rows = []
for tkr, pos in positions.items():
    j = ticker_idx[tkr]
    current_price = close_v[last_i, j]
    open_rows.append({
        'ticker': tkr,
        'entry_date': pos['entry_date'].date(),
        'entry_price': round(pos['entry_price'], 2),
        'current_price': round(current_price, 2) if not np.isnan(current_price) else None,
        'unrealized_pct': round((current_price / pos['entry_price'] - 1) * 100, 1) if not np.isnan(current_price) else None,
        'current_z': round(z_v[last_i, j], 2) if not np.isnan(z_v[last_i, j]) else None,
    })
open_positions_table = pd.DataFrame(open_rows)
if not open_positions_table.empty:
    open_positions_table = open_positions_table.sort_values('unrealized_pct', ascending=False)
display(open_positions_table)

## Trade statistics

Another mean-reversion payoff shape to compare against the RSI pullback notebook: expect a solid
win rate, made mostly of quick, modest wins, against a smaller number of larger losses on the
trades where the "temporary" decline wasn't temporary. With no protective stop, this notebook is
the one where that shape is least defended against.

In [ ]:
trade_stats = pd.Series({
    'total_trades': len(trades),
    'win_rate': float((trades['return_pct'] > 0).mean()),
    'avg_win_pct': float(trades.loc[trades['return_pct'] > 0, 'return_pct'].mean()),
    'avg_loss_pct': float(trades.loc[trades['return_pct'] <= 0, 'return_pct'].mean()),
    'avg_holding_days': float(trades['holding_days'].mean()),
    'best_trade_pct': float(trades['return_pct'].max()),
    'worst_trade_pct': float(trades['return_pct'].min()),
    'profit_factor': float(
        trades.loc[trades['return_pct'] > 0, 'return_pct'].sum()
        / -trades.loc[trades['return_pct'] <= 0, 'return_pct'].sum()
    ),
})
display(trade_stats.to_frame('value'))

### Focus subset: the liquid mega-caps

The names the spec calls out specifically — how did *this* strategy do on just NVDA, AMD, AVGO,
MU, TSM, GOOGL, AMZN, META, and MSFT? Same trades as above, just filtered down.

In [ ]:
focus_trades = trades[trades['ticker'].isin(FOCUS_TICKERS)]
focus_summary = pd.Series({
    'focus_trades': len(focus_trades),
    'focus_win_rate': float((focus_trades['return_pct'] > 0).mean()) if len(focus_trades) else float('nan'),
    'focus_avg_return_pct': float(focus_trades['return_pct'].mean()) if len(focus_trades) else float('nan'),
    'focus_avg_holding_days': float(focus_trades['holding_days'].mean()) if len(focus_trades) else float('nan'),
})
display(focus_summary.to_frame('value'))

by_ticker = focus_trades.groupby('ticker')['return_pct'].agg(['count', 'mean', 'min', 'max']).round(1)
by_ticker.columns = ['trades', 'avg_return_pct', 'worst_pct', 'best_pct']
display(by_ticker.sort_values('avg_return_pct', ascending=False))

## General statistics

Same shape of summary as the breakout and RSI pullback notebooks, computed from the hourly NAV series (annualized with &radic;(252 x 7) bars per year).

In [ ]:
def max_drawdown(wealth):
    drawdown = wealth / wealth.cummax() - 1.0
    return float(drawdown.min())

nav_returns = nav.pct_change().dropna()
start_date, end_date = nav.index[0], nav.index[-1]
years = (end_date - start_date).days / 365.2425
total_return = float(nav.iloc[-1] - 1.0)
cagr = float(nav.iloc[-1] ** (1 / years) - 1.0)
annualized_volatility = float(nav_returns.std() * np.sqrt(BARS_PER_YEAR))
sharpe_rf0 = float((nav_returns.mean() * BARS_PER_YEAR) / annualized_volatility) if annualized_volatility else float('nan')

stats = pd.Series({
    'start': start_date,
    'end': end_date,
    'hourly_bars': len(nav),
    'total_return': total_return,
    'CAGR': cagr,
    'annualized_volatility': annualized_volatility,
    'sharpe_ratio_rf0': sharpe_rf0,
    'max_drawdown': max_drawdown(nav),
    'final_wealth_multiple': float(nav.iloc[-1]),
})
display(stats.to_frame('value'))

## Buy-and-hold-style dollar simulation

Same purchasing-power framing as the other notebooks: size a 2009-era investment so it has the
same purchasing power as $150 today (via FRED CPI), run it through this strategy's compounded
return, and compare the ending value against today's $150 in real terms. Price-only, no-cost
simulation — the Z-score exit is assumed to fill at exactly its trigger price.

In [ ]:
try:
    from pandas_datareader import data as web
    cpi = web.DataReader('CPIAUCSL', 'fred', '2008-12-01', pd.Timestamp.now())['CPIAUCSL'].dropna()
    cpi_2009 = float(cpi.loc[cpi.index >= '2009-01-01'].iloc[0])
    target_cpi = float(TARGET_CPI) if TARGET_CPI is not None else float(cpi.iloc[-1])
    cpi_source = 'FRED CPIAUCSL'
except Exception as exc:
    if TARGET_CPI is None:
        raise RuntimeError('FRED CPI retrieval failed. Set TARGET_CPI to a CPIAUCSL value for the target dollar period.') from exc
    cpi_2009 = 211.143
    target_cpi = float(TARGET_CPI)
    cpi_source = 'manual target CPI; Jan 2009 CPI fallback'

initial_2009_dollars = INVESTMENT_2026_DOLLARS * cpi_2009 / target_cpi
ending_value = initial_2009_dollars * nav.iloc[-1]
nominal_gain = ending_value - initial_2009_dollars
real_gain_vs_target_dollars = ending_value - INVESTMENT_2026_DOLLARS
real_multiple_vs_target_dollars = ending_value / INVESTMENT_2026_DOLLARS

simulation = pd.Series({
    'target_dollars': INVESTMENT_2026_DOLLARS,
    'cpi_2009': cpi_2009,
    'target_cpi': target_cpi,
    'initial_investment_in_2009_dollars': initial_2009_dollars,
    'ending_nominal_value': ending_value,
    'nominal_gain': nominal_gain,
    'total_return': total_return,
    'CAGR': cagr,
    'ending_value_in_target_dollars': ending_value,
    'real_gain_vs_target_dollars': real_gain_vs_target_dollars,
    'real_multiple_vs_target_dollars': real_multiple_vs_target_dollars,
})
print(f'CPI source: {cpi_source}')
display(simulation.to_frame('value'))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
nav.plot(ax=ax, color='#176b87', title='Growth of $1 — Z-score mean reversion in a 200-SMA uptrend, halal universe')
ax.set_ylabel('Portfolio value')
ax.axhline(1, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(13, 3))
open_count = pd.Series(0, index=dates)
for t in trades.itertuples():
    open_count.loc[t.entry_date:t.exit_date] += 1
open_count.plot(ax=ax, color='#d97706', title=f'Concurrent open positions (cap = {MAX_POSITIONS})')
ax.set_ylabel('Positions held')
plt.tight_layout()
plt.show()

## Monthly performance

The hourly NAV series resampled to month-end, so this strategy can be read on the same monthly
heatmap as the other notebooks even though it actually trades on hourly signals.

In [ ]:
monthly_nav = nav.resample('ME').last()
today_ts = pd.Timestamp.now(tz='UTC')
if len(monthly_nav) and (monthly_nav.index[-1].year, monthly_nav.index[-1].month) == (today_ts.year, today_ts.month):
    monthly_nav = monthly_nav.iloc[:-1]
monthly_returns = monthly_nav.pct_change().dropna()

table = (monthly_returns * 100).rename('ret').to_frame()
table['year'] = table.index.year
table['month'] = table.index.strftime('%b')
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_table = table.pivot(index='year', columns='month', values='ret').reindex(columns=month_order)

annual_return = monthly_returns.groupby(monthly_returns.index.year).apply(lambda x: (1 + x).prod() - 1) * 100
monthly_table['Year'] = annual_return

monthly_summary = pd.Series({
    'months': len(monthly_returns),
    'positive_months': int((monthly_returns > 0).sum()),
    'pct_positive_months': float((monthly_returns > 0).mean()),
    'avg_monthly_return': float(monthly_returns.mean()),
    'median_monthly_return': float(monthly_returns.median()),
    'best_month': monthly_returns.idxmax().strftime('%Y-%m'),
    'best_month_return': float(monthly_returns.max()),
    'worst_month': monthly_returns.idxmin().strftime('%Y-%m'),
    'worst_month_return': float(monthly_returns.min()),
})

styled = (
    monthly_table.round(1)
    .style.format(precision=1, na_rep='')
    .background_gradient(cmap='RdYlGn', axis=None, vmin=-15, vmax=15, subset=month_order)
    .set_caption('Monthly return (%) by year')
)
display(styled)
display(monthly_summary.to_frame('value'))

In [ ]:
colors = np.where(monthly_returns >= 0, '#176b87', '#c0392b')
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(monthly_returns.index, monthly_returns.values * 100, width=20, color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Z-score mean reversion monthly return (%)')
ax.set_ylabel('Return (%)')
plt.tight_layout()
plt.show()

## Caveats and next improvements

- **No protective stop, by design** &mdash; and this is the notebook where that matters most. A
  name can keep falling well past Z = -2 (a 2&sigma; move is not a floor), and this strategy has no
  exit until price recovers back above its own 20-day mean, however long that takes or however far
  it falls first. Compare this backtest's max drawdown against the breakout notebook's, which does
  have a hard stop.
- **No transaction costs or slippage**: the Z-score exit is assumed to fill at exactly its trigger
  price, same simplification as the other hourly-signal notebooks.
- **Entry has no confirmation step**: unlike the RSI pullback notebook, this buys directly into the
  extreme reading rather than waiting for a sign of reversal — a deliberate design choice per the
  spec, but it means this strategy is more exposed to a decline that keeps accelerating right after
  entry.
- **Equal-weight-of-NAV sizing**: every one of the 15 slots gets an equal dollar amount regardless
  of how extreme the Z-score is or how volatile the name has been. Sizing is cash-constrained — a
  new entry never spends more than what's actually sitting in cash, even if the 1/15-of-NAV target
  momentarily exceeds it.
- **Same-bar signal and fill**: the Z-score and 200-day SMA checks use today's close, and the trade
  is assumed to fill at that same close.
- **The two "exit" conditions in the original spec are the same rule**: "Z-score > 0" and "price
  returns to its 20-day SMA" aren't two options to choose between - given the Z-score is defined
  against that same 20-day mean, they're mathematically identical.
- **Look-ahead / survivorship bias in the universe**: `equities-list.csv` is today's list of halal
  candidates, projected backward, same caveat as every other notebook in this project.
- **Hourly bars, still price-only and still costless**: hourly bars make signals and marks finer,
  not fills more realistic. Every entry and exit here still fills at the bar's own close (or the
  trigger price), with no spread, slippage or commissions — and an hourly system that reacts to
  more bars trades more, so the costs left out matter more than in the daily version. Look-backs are
  written in trading days and converted at 7 bars per session; half-day sessions have fewer bars,
  so those windows stretch slightly in calendar time.
- A serious version of this strategy would need a hard stop or a maximum holding period (since
  there's currently no limit on how long or how far a losing trade can run), realistic slippage,
  and probably a look at whether the Z-window and SMA-trend window should differ by name given how
  much volatility varies across this universe (see the volatility-adjusted momentum notebook).